# AEMO Data Ingestion and Exploration

This notebook demonstrates the use of the `aemo_data` module for fetching and analyzing Australian Energy Market Operator (AEMO) datasets.

## Data Sources

The AEMO data module uses **NEMOSIS** (https://github.com/UNSW-CEEM/NEMOSIS) to fetch **actual** AEMO market data, including:
- **Energy Prices**: Regional Reference Price (RRP) and total demand at 5-minute intervals
- **FCAS Prices**: Frequency Control Ancillary Services prices for maintaining grid stability (regional)
- **Generation Mix**: Power output by fuel type (solar, wind, coal, gas, etc.)
- **Unit Dispatch**: Unit-specific dispatch targets and FCAS enablement (NEW)

### NEMOSIS Integration

NEMOSIS automatically:
- Downloads data from AEMO's NEMWEB archives
- Caches data locally for faster subsequent access
- Handles data parsing and formatting

AEMO publishes market data through:
- **NEMWEB**: http://nemweb.com.au/ - Historical market data archive
- **MMS Data Model**: Detailed pricing and generation records

### Important Notes

- **First run**: May take 1-2 minutes to download data from AEMO
- **Cached runs**: Much faster as data is loaded from local cache
- **Internet required**: NEMOSIS needs connectivity to download from AEMO servers
- **Data availability**: Use dates from 2-3 months ago for best reliability (recent data may still be preliminary)
- **Data resolution**: 5-minute dispatch intervals (288 intervals per day)

In [1]:
import sys
sys.path.append('src')

from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import polars as pl
import numpy as np

from aemo_data import (
    fetch_aemo_dispatch_price,
    fetch_aemo_fcas_price,
    fetch_aemo_generation_by_fuel,
    fetch_aemo_unit_dispatch,
    fetch_aemo_data_bundle,
    fetch_aemo_data_bundle_with_dispatch,
    AEMO_REGIONS,
    FCAS_SERVICES,
)

In [2]:
print("Available AEMO Regions:")
print(AEMO_REGIONS)
print("\nAvailable FCAS Services:")
print(FCAS_SERVICES)

Available AEMO Regions:
['NSW1', 'QLD1', 'SA1', 'TAS1', 'VIC1']

Available FCAS Services:
['RAISE6SEC', 'LOWER6SEC', 'RAISE60SEC', 'LOWER60SEC', 'RAISE5MIN', 'LOWER5MIN', 'RAISEREG', 'LOWERREG']


## Sample DUIDs for Testing

Here are some commonly used DUIDs across different regions and technologies:

### Battery Energy Storage Systems (BESS)
- **HPRG1** - Hornsdale Power Reserve (SA) - 150 MW / 194 MWh
- **LBBG1** - Lake Bonney Battery (SA) - 25 MW / 52 MWh
- **DALNTH01** - Dalrymple North BESS (SA) - 30 MW / 8 MWh
- **GANNBG1** - Gannawarra BESS (VIC) - 25 MW / 50 MWh
- **BALBG1** - Ballarat BESS (VIC) - 30 MW / 30 MWh

### Solar Farms
- **SUNSH1** - Sunraysia Solar Farm (VIC) - 100 MW
- **NYNGAN1** - Nyngan Solar Plant (NSW) - 102 MW
- **BROKENH1** - Broken Hill Solar Plant (NSW) - 53 MW
- **CLARESF1** - Clare Solar Farm (QLD) - 100 MW

### Wind Farms
- **SNOWTWN1** - Snowtown Wind Farm (SA) - 129 MW
- **NBHWF1** - North Brown Hill Wind Farm (SA) - 132 MW
- **MACARTH1** - Macarthur Wind Farm (VIC) - 420 MW
- **CAPTL_WF** - Capital Wind Farm (NSW) - 140 MW

### Gas Generators
- **PPCCGT** - Pelican Point CCGT (SA) - 478 MW
- **TORRB1** - Torrens Island B (SA) - 800 MW
- **VPGS1** - Valley Power Peaking Station (VIC) - 300 MW

### Hydro
- **TUMMER1** - Tumut 3 (NSW) - 1500 MW
- **MURRAY** - Murray (NSW) - 1500 MW
- **GUTHEGA** - Guthega (NSW) - 60 MW

**Note**: DUID availability varies by date. Use dates when the unit was operational.

## 1. Fetch Regional Market Data

Let's fetch 12 hours of actual AEMO data from June 2023. Using historical data ensures reliability as it's fully archived.

In [3]:
# Define date range (using historical data for reliability)
# Use data from several months ago as it's fully archived and stable
start_date = datetime(2023, 6, 1, 0, 0, 0)
end_date = datetime(2023, 6, 1, 12, 0, 0)  # 12 hours of data
region = AEMO_REGIONS[-1]

print(f"Fetching ACTUAL AEMO data from {start_date} to {end_date}")
print(f"Region: {region}")
print("\nNote: First run will download data from AEMO (1-2 minutes)")
print("      Subsequent runs use cached data (much faster)")

Fetching ACTUAL AEMO data from 2023-06-01 00:00:00 to 2023-06-01 12:00:00
Region: VIC1

Note: First run will download data from AEMO (1-2 minutes)
      Subsequent runs use cached data (much faster)


### 1.1 Energy Prices

In [4]:
# Fetch energy prices
prices = fetch_aemo_dispatch_price(start_date, end_date, region=region)

print("\nEnergy Price Data:")
print(prices.head())
print(f"\nTotal records: {len(prices)}")
print(f"\nPrice statistics ($/MWh):")
print(prices.select(["RRP", "TOTALDEMAND"]).describe())

Fetching dispatch price data for VIC1 from 2023-06-01 to 2023-06-01...
INFO: Compiling data for table DISPATCHPRICE
INFO: Returning DISPATCHPRICE.
INFO: Compiling data for table DISPATCHREGIONSUM
INFO: Returning DISPATCHREGIONSUM.
Fetched 144 price records

Energy Price Data:
shape: (5, 4)
┌─────────────────────┬──────────┬───────────┬─────────────┐
│ SETTLEMENTDATE      ┆ REGIONID ┆ RRP       ┆ TOTALDEMAND │
│ ---                 ┆ ---      ┆ ---       ┆ ---         │
│ datetime[ns]        ┆ str      ┆ f64       ┆ f64         │
╞═════════════════════╪══════════╪═══════════╪═════════════╡
│ 2023-06-01 00:05:00 ┆ VIC1     ┆ 0.02      ┆ 4475.86     │
│ 2023-06-01 00:10:00 ┆ VIC1     ┆ -29.26    ┆ 4423.25     │
│ 2023-06-01 00:15:00 ┆ VIC1     ┆ -30.61    ┆ 4433.47     │
│ 2023-06-01 00:20:00 ┆ VIC1     ┆ -46.74    ┆ 4326.08     │
│ 2023-06-01 00:25:00 ┆ VIC1     ┆ -47.45009 ┆ 4257.42     │
└─────────────────────┴──────────┴───────────┴─────────────┘

Total records: 144

Price statistics 

### 1.2 FCAS Prices (Regional)

In [5]:
# Fetch FCAS prices for regulation services
fcas_raise = fetch_aemo_fcas_price(start_date, end_date, region=region, service="RAISEREG")
fcas_lower = fetch_aemo_fcas_price(start_date, end_date, region=region, service="LOWERREG")

print("\nFCAS Raise Regulation Prices:")
print(fcas_raise.head())

print("\nFCAS Price Statistics ($/MW/h):")
print(fcas_raise.select(["PRICE"]).describe())

Fetching FCAS RAISEREG price data for VIC1 from 2023-06-01 to 2023-06-01...
INFO: Compiling data for table DISPATCHPRICE
INFO: Returning DISPATCHPRICE.
Fetched 144 FCAS price records
Fetching FCAS LOWERREG price data for VIC1 from 2023-06-01 to 2023-06-01...
INFO: Compiling data for table DISPATCHPRICE
INFO: Returning DISPATCHPRICE.
Fetched 144 FCAS price records

FCAS Raise Regulation Prices:
shape: (5, 4)
┌─────────────────────┬──────────┬──────────┬───────┐
│ SETTLEMENTDATE      ┆ REGIONID ┆ SERVICE  ┆ PRICE │
│ ---                 ┆ ---      ┆ ---      ┆ ---   │
│ datetime[ns]        ┆ str      ┆ str      ┆ f64   │
╞═════════════════════╪══════════╪══════════╪═══════╡
│ 2023-06-01 00:05:00 ┆ VIC1     ┆ RAISEREG ┆ 6.0   │
│ 2023-06-01 00:10:00 ┆ VIC1     ┆ RAISEREG ┆ 5.0   │
│ 2023-06-01 00:15:00 ┆ VIC1     ┆ RAISEREG ┆ 12.0  │
│ 2023-06-01 00:20:00 ┆ VIC1     ┆ RAISEREG ┆ 7.99  │
│ 2023-06-01 00:25:00 ┆ VIC1     ┆ RAISEREG ┆ 7.5   │
└─────────────────────┴──────────┴──────────┴────

### 1.3 Generation by Fuel Type

In [8]:
# TODO: Find the excel file that was cached in data/aemo and pass its path to the function below

In [ ]:
# Fetch generation data
generation = fetch_aemo_generation_by_fuel(
    start_date, end_date, 
    region=region, 
    fuel_types=["solar", "wind"],
    refresh=True,
)

print("\nGeneration Data:")
print(generation.head())
print(f"\nTotal records: {len(generation)}")

Fetching generation data for ['solar', 'wind'] in NSW1 from 2023-06-01 to 2023-06-01...
INFO: Retrieving static table Generators and Scheduled Loads
Error fetching generation data from NEMOSIS: Excel reading library (openpyxl) is required to fetch generator information. Install with: pip install openpyxl
Note: NEMOSIS requires internet connection to download data from AEMO


ImportError: Excel reading library (openpyxl) is required to fetch generator information. Install with: pip install openpyxl

## 2. Fetch Unit-Specific Dispatch Data (NEW)

The new `fetch_aemo_unit_dispatch()` function fetches unit-specific dispatch targets and FCAS enablement from the DISPATCHLOAD table. This is essential for calculating actual FCAS revenue and understanding operational constraints.

### 2.1 Fetch Dispatch Data for a Specific Battery

In [ ]:
# Fetch dispatch data for Hornsdale Power Reserve (HPRG1) - SA's big battery
unit_dispatch = fetch_aemo_unit_dispatch(
    start_date=datetime(2023, 6, 1, 0, 0, 0),
    end_date=datetime(2023, 6, 1, 6, 0, 0),  # 6 hours
    duid="HPRG1",
    region="SA1"
)

print("\nUnit Dispatch Data for HPRG1:")
print(unit_dispatch.head())
print(f"\nTotal dispatch records: {len(unit_dispatch)}")
print(f"\nColumns available: {unit_dispatch.columns}")

### 2.2 Analyze FCAS Enablement

In [ ]:
# Check which FCAS services the battery was enabled for
if len(unit_dispatch) > 0:
    fcas_cols = [col for col in unit_dispatch.columns if 'ACTUALAVAILABILITY' in col]
    
    print("\nFCAS Enablement Summary:")
    for col in fcas_cols:
        service_name = col.replace('ACTUALAVAILABILITY', '')
        enablement_data = unit_dispatch.select([col])
        max_enabled = enablement_data[col].max()
        avg_enabled = enablement_data[col].mean()
        print(f"{service_name:20s}: Max={max_enabled:6.1f} MW, Avg={avg_enabled:6.1f} MW")

### 2.3 Calculate FCAS Revenue

Revenue = Enablement (MW) × Price ($/MW/h) × interval duration (hours)

In [ ]:
# Fetch FCAS prices for SA1
fcas_prices_sa = fetch_aemo_fcas_price(
    datetime(2023, 6, 1, 0, 0, 0),
    datetime(2023, 6, 1, 6, 0, 0),
    region="SA1",
    service="RAISEREG"
)

# Join enablement with prices (simplified example)
if len(unit_dispatch) > 0 and len(fcas_prices_sa) > 0:
    # Convert to pandas for easier joining
    dispatch_pd = unit_dispatch.to_pandas()
    prices_pd = fcas_prices_sa.to_pandas()
    
    # Merge on settlement date
    merged = dispatch_pd.merge(prices_pd, on='SETTLEMENTDATE', how='inner')
    
    if len(merged) > 0 and 'RAISEREGACTUALAVAILABILITY' in merged.columns:
        # Calculate revenue (5-minute interval = 1/12 hour)
        merged['RAISEREG_REVENUE'] = merged['RAISEREGACTUALAVAILABILITY'] * merged['PRICE'] * (5/60)
        
        total_revenue = merged['RAISEREG_REVENUE'].sum()
        print(f"\nEstimated FCAS RAISEREG Revenue for 6 hours: ${total_revenue:,.2f}")
        print(f"Average revenue per interval: ${merged['RAISEREG_REVENUE'].mean():.2f}")

## 3. Comprehensive Data Bundle with Dispatch

Use the new `fetch_aemo_data_bundle_with_dispatch()` wrapper to fetch all market data in one call.

In [ ]:
# Fetch comprehensive data for a battery unit
data_bundle = fetch_aemo_data_bundle_with_dispatch(
    start_date=datetime(2023, 6, 1, 0, 0, 0),
    end_date=datetime(2023, 6, 1, 3, 0, 0),  # 3 hours
    region="SA1",
    duid="HPRG1",
    fcas_services=["RAISEREG", "LOWERREG"],
    fuel_types=["solar", "wind"]
)

print("\nData Bundle Contents:")
print(f"Energy prices: {len(data_bundle['prices'])} records")
print(f"FCAS prices: {len(data_bundle['fcas'])} records")
print(f"Generation: {len(data_bundle['generation'])} records")
print(f"Unit dispatch: {len(data_bundle['unit_dispatch'])} records")

print("\nUnit Dispatch Sample:")
print(data_bundle['unit_dispatch'].head())

## 4. Explore Different DUIDs

Let's try fetching data for different unit types to see what's available.

In [ ]:
# Test different DUIDs
test_duids = [
    ("HPRG1", "SA1", "Battery - Hornsdale"),
    ("LBBG1", "SA1", "Battery - Lake Bonney"),
    ("SUNSH1", "VIC1", "Solar - Sunraysia"),
    ("SNOWTWN1", "SA1", "Wind - Snowtown"),
]

print("Testing dispatch data availability for different units:\n")

for duid, region, description in test_duids:
    try:
        dispatch = fetch_aemo_unit_dispatch(
            start_date=datetime(2023, 6, 1, 0, 0, 0),
            end_date=datetime(2023, 6, 1, 1, 0, 0),  # 1 hour
            duid=duid,
            region=region
        )
        print(f"✓ {duid:12s} ({description:25s}): {len(dispatch)} records")
    except Exception as e:
        print(f"✗ {duid:12s} ({description:25s}): Error - {str(e)[:50]}")

## 5. Visualizations

In [ ]:
# Plot energy and FCAS prices
if len(prices) > 0 and len(fcas_raise) > 0:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Energy prices
    prices_pd = prices.to_pandas()
    ax1.plot(prices_pd['SETTLEMENTDATE'], prices_pd['PRICE'], label='Energy Price', color='blue')
    ax1.set_ylabel('Price ($/MWh)')
    ax1.set_title('AEMO Energy Prices - NSW1')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # FCAS prices
    fcas_pd = fcas_raise.to_pandas()
    ax2.plot(fcas_pd['SETTLEMENTDATE'], fcas_pd['PRICE'], label='FCAS Raise Reg', color='green')
    ax2.set_ylabel('Price ($/MW/h)')
    ax2.set_xlabel('Time')
    ax2.set_title('FCAS Regulation Raise Prices - NSW1')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot unit dispatch and FCAS enablement
if len(unit_dispatch) > 0:
    dispatch_pd = unit_dispatch.to_pandas()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Energy dispatch
    if 'TOTALCLEARED' in dispatch_pd.columns:
        ax1.plot(dispatch_pd['SETTLEMENTDATE'], dispatch_pd['TOTALCLEARED'], label='Energy Dispatch', color='blue')
        ax1.set_ylabel('Power (MW)')
        ax1.set_title('Unit Energy Dispatch - HPRG1')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    
    # FCAS enablement
    fcas_cols = [col for col in dispatch_pd.columns if 'ACTUALAVAILABILITY' in col]
    for col in fcas_cols[:4]:  # Plot first 4 FCAS services
        service_name = col.replace('ACTUALAVAILABILITY', '')
        ax2.plot(dispatch_pd['SETTLEMENTDATE'], dispatch_pd[col], label=service_name, alpha=0.7)
    
    ax2.set_ylabel('Enablement (MW)')
    ax2.set_xlabel('Time')
    ax2.set_title('FCAS Enablement - HPRG1')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Summary

This notebook demonstrated:

1. **Regional Market Data**: Fetching energy prices, FCAS prices, and generation mix
2. **Unit-Specific Dispatch**: NEW function to fetch dispatch targets and FCAS enablement for individual units
3. **FCAS Revenue Calculation**: How to join enablement with prices to calculate actual revenue
4. **Comprehensive Wrapper**: Using `fetch_aemo_data_bundle_with_dispatch()` for all data in one call
5. **Different Unit Types**: Testing various DUIDs (batteries, solar, wind, etc.)

### Key Takeaways

- Regional FCAS prices tell you what the market is paying
- Unit-specific enablement tells you what capacity was actually dispatched
- Revenue = Enablement (MW) × Price ($/MW/h) × interval duration
- Different unit types have different participation patterns in FCAS markets

### Next Steps

- Use this data to train RL agents for battery trading optimization
- Analyze correlations between renewable generation and FCAS prices
- Build reward functions that incorporate both energy arbitrage and FCAS revenue